# Métricas de manzana desde las huellas de edificio

Cuatro métricas que separan **trama espontánea vs planificada**, calculadas sobre los **bloques**
(manzanas edificadas: fusión por cierre morfológico de las huellas que se tocan, patios rellenos,
contorno simplificado).

**Regularidad del contorno**
- `edgeR4` — R4 (4º armónico) de los rumbos de los **lados** del bloque. Rectángulo = 2 direcciones
  perpendiculares → ≈1; contorno irregular → muchas direcciones → bajo.
- `rectangularity` — área / área del rectángulo rotado mínimo. 1 = rectángulo perfecto.

**Homogeneidad local** (¿se parece la manzana a sus 6 vecinas más cercanas?)
- `area_local_cv` — coeficiente de variación de las áreas del vecindario. BAJO = áreas parecidas (planificado).
- `angle_align` — alineación circular de la orientación de rejilla (fase de R4, ponderada por `edgeR4`)
  con las vecinas. ALTO = misma orientación (planificado).

Validado: Centro histórico vs Ensanche → edgeR4 0.73/0.93 · rect 0.75/0.88 · area_local_cv 0.69/0.46 · angle_align 0.73/0.96.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, geopandas as gpd, matplotlib.pyplot as plt
from shapely import unary_union
from shapely.geometry import Polygon, box
from scipy.spatial import cKDTree
from pathlib import Path

BASE    = Path(r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA')
CAT_SHP = BASE / 'Catastro' / 'catastro_comunidad_de_madrid.shp'
GPKG    = BASE / 'madrid_morfologia_h3_v10.gpkg'
CRS     = 'EPSG:25830'
POZUELO = ['Aravaca', 'El Plantío', 'Valdemarín']
OUTDIR  = Path('imagenes'); OUTDIR.mkdir(exist_ok=True)
CACHE   = Path('bloques_huellas.gpkg')   # se reutiliza si ya existe

CLOSE, SIMP, MIN_BLK, K = 2.0, 3.0, 50, 6
ZOOM = (438700, 4473000, 442900, 4476500)   # centro histórico (O) ↔ Ensanche (E)
FEATS = ['edgeR4', 'rectangularity', 'area_local_cv', 'angle_align']

## Helpers de forma y orientación

In [ ]:
def edge_r4(poly):
    """R4 de los rumbos de los lados (ponderado por longitud). Rectángulo→≈1."""
    xy = np.asarray(poly.exterior.coords); s = np.diff(xy, axis=0); L = np.hypot(s[:, 0], s[:, 1])
    if L.sum() == 0: return np.nan
    th = np.arctan2(s[:, 1], s[:, 0]); w = L / L.sum()
    return float(abs(np.sum(w * np.exp(1j * 4 * th))))

def rectangularity(poly):
    mrr = poly.minimum_rotated_rectangle
    return poly.area / mrr.area if mrr.area > 0 else np.nan

def edge_z(poly):
    """Vector complejo R4 (módulo=edgeR4, fase=orientación de rejilla)."""
    xy = np.asarray(poly.exterior.coords); s = np.diff(xy, axis=0); L = np.hypot(s[:, 0], s[:, 1])
    if L.sum() == 0: return 0j
    th = np.arctan2(s[:, 1], s[:, 0]); w = L / L.sum()
    return np.sum(w * np.exp(1j * 4 * th))

## 1. Construir los bloques y calcular las 4 métricas (o cargar de caché)

In [ ]:
if CACHE.exists() and set(FEATS).issubset(set(gpd.read_file(CACHE, rows=1).columns)):
    bdf = gpd.read_file(CACHE)
    print(f'Cargado de caché: {len(bdf):,} bloques')
else:
    print('Cargando catastro…')
    try:
        import pyogrio
        cat = pyogrio.read_dataframe(str(CAT_SHP), columns=['gml_id']).to_crs(CRS)
    except Exception:
        cat = gpd.read_file(str(CAT_SHP))[['geometry']].to_crs(CRS)
    cat = cat[cat.geometry.notna() & cat.geometry.is_valid]
    hexg = gpd.read_file(GPKG).to_crs(CRS); hexg = hexg[~hexg['barrio_nombre'].isin(POZUELO)]
    city = hexg.geometry.union_all() if hasattr(hexg.geometry, 'union_all') else hexg.geometry.unary_union
    cat = cat[cat.geometry.centroid.within(city)].copy()
    print(f'  huellas en la ciudad: {len(cat):,}')

    print(f'Fusionando huellas (CLOSE={CLOSE} m)…')
    merged = unary_union(cat.geometry.buffer(CLOSE)).buffer(-CLOSE)
    parts = list(merged.geoms) if merged.geom_type == 'MultiPolygon' else [merged]
    blocks = []
    for g in parts:
        if g.is_empty or g.area < MIN_BLK: continue
        env = Polygon(g.exterior).simplify(SIMP, preserve_topology=True)
        if env.is_valid and env.area >= MIN_BLK: blocks.append(env)
    bdf = gpd.GeoDataFrame(geometry=gpd.GeoSeries(blocks, crs=CRS))

    # contorno
    bdf['edgeR4'] = bdf.geometry.apply(edge_r4)
    bdf['rectangularity'] = bdf.geometry.apply(rectangularity)
    # homogeneidad local (K vecinos más cercanos)
    z = np.array([edge_z(g) for g in bdf.geometry]); mag = np.abs(z)
    unit = np.where(mag > 0, z / np.maximum(mag, 1e-9), 0)
    A = bdf.geometry.area.values
    cen = np.c_[bdf.geometry.centroid.x, bdf.geometry.centroid.y]
    _, idx = cKDTree(cen).query(cen, k=K + 1)
    acv = np.full(len(bdf), np.nan); al = np.full(len(bdf), np.nan)
    for i in range(len(bdf)):
        nb = idx[i]; a = A[nb]
        acv[i] = a.std() / a.mean() if a.mean() > 0 else np.nan
        w = mag[nb]; al[i] = abs(np.sum(unit[nb] * w) / w.sum()) if w.sum() > 0 else np.nan
    bdf['area_local_cv'] = acv; bdf['angle_align'] = al
    bdf.to_file(CACHE)
    print(f'  bloques: {len(bdf):,}  → cacheado en {CACHE}')

print(bdf[FEATS].describe().round(3).to_string())

## 2. Comprobación: centro histórico vs Ensanche

In [ ]:
for name, w in {'Centro histórico': (439400, 4473300, 440200, 4474100),
                'Ensanche (Salamanca)': (441100, 4474600, 441900, 4475400)}.items():
    s = bdf[bdf.intersects(box(*w))]
    print(f'  {name:22s}: edgeR4={s.edgeR4.median():.2f}  rect={s.rectangularity.median():.2f}  '
          f'area_local_cv={s.area_local_cv.median():.2f}  angle_align={s.angle_align.median():.2f}')

## 3. Mapas (ciudad + zoom)

In [ ]:
def panel_map(panels, fname, suptitle):
    bz = bdf[bdf.intersects(box(*ZOOM))]
    fig, axes = plt.subplots(2, len(panels), figsize=(10 * len(panels), 18))
    for col, (var, cmap, (vmin, vmax), tit, sub) in enumerate(panels):
        ax = axes[0, col]
        bdf.plot(column=var, cmap=cmap, vmin=vmin, vmax=vmax, linewidth=0, ax=ax,
                 legend=True, legend_kwds={'orientation': 'horizontal', 'shrink': 0.5, 'pad': 0.01})
        ax.set_title(f'{tit}\n{sub} — ciudad', fontsize=12); ax.set_axis_off()
        ax = axes[1, col]
        bz.plot(column=var, cmap=cmap, vmin=vmin, vmax=vmax, linewidth=0.2, edgecolor='white', ax=ax)
        ax.set_xlim(ZOOM[0], ZOOM[2]); ax.set_ylim(ZOOM[1], ZOOM[3])
        ax.set_title(f'{tit} — zoom centro/Ensanche', fontsize=12); ax.set_axis_off()
    fig.suptitle(suptitle, fontsize=16, y=1.0); plt.tight_layout()
    plt.savefig(OUTDIR / fname, dpi=140, bbox_inches='tight'); print(f'Guardado: {OUTDIR / fname}'); plt.show()

In [ ]:
# Contorno
panel_map(
    [('edgeR4', 'RdYlBu', (0.3, 1.0), 'edgeR4 — ortogonalidad del contorno', 'bajo = irregular (espontáneo)'),
     ('rectangularity', 'RdYlBu', (0.4, 1.0), 'Rectangularidad — área/bbox', 'bajo = irregular (espontáneo)')],
    'manzana_contorno.png', 'Regularidad del contorno de manzana — Madrid (rojo ≈ espontáneo)')

In [ ]:
# Homogeneidad local
panel_map(
    [('area_local_cv', 'RdYlBu_r', (0.2, 1.0), 'Heterogeneidad de área local (CV, K=6)', 'alto = áreas dispares (espontáneo)'),
     ('angle_align', 'RdYlBu', (0.5, 1.0), 'Alineación de orientación local (R4)', 'bajo = orientaciones dispares (espontáneo)')],
    'manzana_homogeneidad.png', 'Homogeneidad local de las manzanas — Madrid (rojo ≈ espontáneo)')

## 4. Agregación a hexágono H3 (media de los bloques por celda)

In [ ]:
import pandas as pd
hexg = gpd.read_file(GPKG).to_crs(CRS)
hexg = hexg[~hexg['barrio_nombre'].isin(POZUELO)].reset_index(drop=True)

# asignar cada bloque a su hexágono por el CENTROIDE del bloque
bc = bdf[FEATS].copy(); bc['geometry'] = bdf.geometry.centroid
bc = gpd.GeoDataFrame(bc, geometry='geometry', crs=CRS)
j = gpd.sjoin(bc, hexg[['hex_id', 'geometry']], predicate='within', how='inner')
agg = j.groupby('hex_id')[FEATS].mean()
agg['n_blocks'] = j.groupby('hex_id').size()
hexg = hexg.merge(agg, on='hex_id', how='left')
print(f'Hexágonos con ≥1 bloque: {hexg["edgeR4"].notna().sum()} de {len(hexg)}  '
      f'(n_blocks mediana = {hexg["n_blocks"].median():.0f})')

## 5. Mapas por hexágono (gris = poco edificado o < 2 bloques)

In [ ]:
MIN_BLD_HEX = 5
urban = ((pd.to_numeric(hexg['cat_n_buildings'], errors='coerce').fillna(0) >= MIN_BLD_HEX)
         & (hexg['n_blocks'] >= 2))
hm = hexg.copy(); hm.loc[~urban, FEATS] = np.nan
print(f'Hexágonos urbanos mostrados: {int(urban.sum())} de {len(hm)}')

def hex_map(panels, fname, suptitle):
    fig, axes = plt.subplots(1, len(panels), figsize=(11 * len(panels), 11))
    axes = np.atleast_1d(axes)
    for ax, (var, cmap, (vmin, vmax), tit, sub) in zip(axes, panels):
        hm.plot(column=var, cmap=cmap, vmin=vmin, vmax=vmax, linewidth=0.05, edgecolor='white', ax=ax,
                legend=True, legend_kwds={'orientation': 'horizontal', 'shrink': 0.6, 'pad': 0.02},
                missing_kwds={'color': 'lightgrey'})
        ax.set_title(f'{tit}  ·  {sub}', fontsize=11); ax.set_axis_off()
    fig.suptitle(suptitle, fontsize=15, y=1.0); plt.tight_layout()
    plt.savefig(OUTDIR / fname, dpi=150, bbox_inches='tight'); print('Guardado:', OUTDIR / fname); plt.show()


In [ ]:
# Contorno (por hexágono)
hex_map(
    [('edgeR4', 'RdYlBu', (0.4, 1.0), 'edgeR4 — ortogonalidad del contorno', 'bajo = irregular (espontáneo)'),
     ('rectangularity', 'RdYlBu', (0.45, 0.95), 'Rectangularidad — área/bbox', 'bajo = irregular (espontáneo)')],
    'hex_manzana_contorno.png', 'Regularidad del contorno de manzana por hexágono H3 — Madrid (rojo ≈ espontáneo)')

In [ ]:
# Homogeneidad local (por hexágono)
hex_map(
    [('area_local_cv', 'RdYlBu_r', (0.3, 0.9), 'Heterogeneidad de área local (CV, K=6)', 'alto = áreas dispares (espontáneo)'),
     ('angle_align', 'RdYlBu', (0.6, 1.0), 'Alineación de orientación local (R4)', 'bajo = orientaciones dispares (espontáneo)')],
    'hex_manzana_homogeneidad.png', 'Homogeneidad local de las manzanas por hexágono H3 — Madrid (rojo ≈ espontáneo)')

## 6. Cohen's d sobre las etiquetas manuales (por hexágono)

In [ ]:
lab = pd.read_csv(BASE / 'labels_hex_manual.csv'); lab['hex_id'] = lab['hex_id'].astype(str)
hexg['label'] = hexg['hex_id'].astype(str).map(
    dict(zip(lab['hex_id'], pd.to_numeric(lab['tipologia_train'], errors='coerce'))))
esp, plan = hexg['label'] == 0, hexg['label'] == 1
print("Cohen's d (espontáneo − planificado), métricas de manzana agregadas por hexágono:")
for v in FEATS:
    c = pd.to_numeric(hexg[v], errors='coerce'); s = c[esp | plan].std()
    d = (c[esp].mean() - c[plan].mean()) / s if s > 0 else float('nan')
    print(f'  {v:16s} d = {d:+.2f}   (cobertura {c.notna().mean()*100:.0f}%)')

In [ ]:
# Manual training labels on the H3 grid
import geopandas as gpd, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

BASE = Path(r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA')

hexg = gpd.read_file(BASE / 'madrid_morfologia_h3_v10.gpkg').to_crs('EPSG:25830')
lab  = pd.read_csv(BASE / 'labels_hex_manual (4).csv').set_index('hex_id')['tipologia_train']

hexg['lab']   = hexg['hex_id'].astype(str).map(lab)                 # NaN where unlabelled
hexg['color'] = hexg['lab'].map({0: '#d7191c', 1: '#2b83ba'}).fillna('#eeeeee')

fig, ax = plt.subplots(figsize=(11, 11))
hexg.plot(color=hexg['color'], edgecolor='white', linewidth=0.1, ax=ax)
ax.legend(handles=[mpatches.Patch(color='#d7191c', label='Spontaneous (0)'),
                   mpatches.Patch(color='#2b83ba', label='Planned (1)'),
                   mpatches.Patch(color='#eeeeee', label='Unlabelled')],
          loc='lower right', fontsize=10)
n0 = int((hexg.lab == 0).sum()); n1 = int((hexg.lab == 1).sum())
ax.set_title(f'Manual training labels (H3): {n1} planned · {n0} spontaneous', fontsize=13)
ax.set_axis_off(); plt.tight_layout(); plt.show()